# D2Q9 Lattice Verification Notebook

This notebook verifies the D2Q9 lattice implementation and demonstrates its functionality.

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lattice import D2Q9Lattice, compute_vorticity

print("✓ Imports successful")

## TEST 1: Lattice Initialization

In [ ]:
print("="*60)
print("TEST 1: Lattice Initialization")
print("="*60)

lattice = D2Q9Lattice()

# Check dimensions
print(f"\n✓ Lattice created: {lattice}")
print(f"  Velocity directions shape: {lattice.c.shape}")
print(f"  Weights shape: {lattice.w.shape}")
print(f"  Speed of sound: cs = {lattice.cs:.6f}, cs² = {lattice.cs2:.6f}")

# Display velocity directions
print("\nVelocity directions:")
for i in range(9):
    print(f"  c[{i}] = ({lattice.c[i,0]:2.0f}, {lattice.c[i,1]:2.0f})  "
          f"weight = {lattice.w[i]:.6f}")

## TEST 2: Weight Normalization

In [ ]:
print("\n" + "="*60)
print("TEST 2: Weight Normalization")
print("="*60)

weight_sum = np.sum(lattice.w)
error = abs(weight_sum - 1.0)

print(f"Sum of weights: {weight_sum:.15f}")
print(f"Error from 1.0: {error:.2e}")
print(f"Status: {'✓ PASS' if error < 1e-12 else '✗ FAIL'}")

# Verify individual weight values
expected_weights = {
    0: 4/9,   # rest
    1: 1/9,   # cardinal
    5: 1/36   # diagonal
}

print("\nIndividual weight verification:")
for idx, expected in expected_weights.items():
    actual = lattice.w[idx]
    error = abs(actual - expected)
    status = "✓" if error < 1e-12 else "✗"
    print(f"  {status} w[{idx}] = {actual:.10f} (expected {expected:.10f})")

## TEST 3: Opposite Direction Mapping

In [ ]:
print("\n" + "="*60)
print("TEST 3: Opposite Direction Mapping")
print("="*60)

print("\nOpposite direction pairs:")
all_pass = True
for i in range(9):
    opp_i = lattice.opp[i]
    sum_vel = lattice.c[i] + lattice.c[opp_i]
    is_zero = np.allclose(sum_vel, [0, 0])
    all_pass = all_pass and is_zero
    status = "✓" if is_zero else "✗"
    print(f"  {status} c[{i}] + c[{opp_i}] = ({sum_vel[0]:2.0f}, {sum_vel[1]:2.0f})")

print(f"\nOverall: {'✓ PASS' if all_pass else '✗ FAIL'}")

## TEST 4: Equilibrium Distribution

In [ ]:
print("\n" + "="*60)
print("TEST 4: Equilibrium Distribution")
print("="*60)

# Create test fields
Nx, Ny = 10, 10
rho = np.ones((Nx, Ny))
u = 0.1 * np.ones((Nx, Ny))
v = 0.05 * np.ones((Nx, Ny))

print(f"\nTest field size: {Nx} x {Ny}")
print(f"Density: {rho[0,0]:.2f}")
print(f"Velocity: u = {u[0,0]:.2f}, v = {v[0,0]:.2f}")

# Compute equilibrium
f_eq = lattice.equilibrium(rho, u, v)

print(f"\n✓ Equilibrium computed, shape: {f_eq.shape}")

# Verify mass conservation
rho_computed = np.sum(f_eq, axis=2)
mass_error = np.max(np.abs(rho_computed - rho))
print(f"\nMass conservation error: {mass_error:.2e}")
print(f"Status: {'✓ PASS' if mass_error < 1e-10 else '✗ FAIL'}")

## TEST 5: Macroscopic Quantity Recovery

In [ ]:
print("\n" + "="*60)
print("TEST 5: Macroscopic Quantity Recovery")
print("="*60)

# Recover macroscopic quantities from equilibrium
rho_out, u_out, v_out = lattice.compute_macroscopic(f_eq)

rho_error = np.max(np.abs(rho_out - rho))
u_error = np.max(np.abs(u_out - u))
v_error = np.max(np.abs(v_out - v))

print(f"\nRecovery errors:")
print(f"  Density error: {rho_error:.2e}")
print(f"  u-velocity error: {u_error:.2e}")
print(f"  v-velocity error: {v_error:.2e}")

all_pass = (rho_error < 1e-10 and u_error < 1e-10 and v_error < 1e-10)
print(f"\nStatus: {'✓ PASS' if all_pass else '✗ FAIL'}")

## TEST 6: Vorticity Computation

In [ ]:
print("\n" + "="*60)
print("TEST 6: Vorticity Computation")
print("="*60)

# Create a simple vortex flow: u = -y, v = x
Nx, Ny = 20, 20
x = np.linspace(-1, 1, Nx)
y = np.linspace(-1, 1, Ny)
X, Y = np.meshgrid(x, y, indexing='ij')

u = -Y
v = X

# Compute vorticity
omega = compute_vorticity(u, v, dx=x[1]-x[0])

# For this flow, vorticity should be constant = 2
expected_vorticity = 2.0
interior = omega[2:-2, 2:-2]
mean_vorticity = np.mean(interior)
vorticity_error = abs(mean_vorticity - expected_vorticity)

print(f"\nExpected vorticity: {expected_vorticity:.2f}")
print(f"Mean vorticity (interior): {mean_vorticity:.4f}")
print(f"Error: {vorticity_error:.4f}")
print(f"Status: {'✓ PASS' if vorticity_error < 0.1 else '✗ FAIL'}")

## Visualization: Lattice Velocity Directions

In [ ]:
# Visualize lattice directions
fig, ax = plt.subplots(figsize=(8, 8))

# Plot velocity vectors
origin = [0, 0]
for i in range(9):
    if i == 0:
        # Rest particle - plot as a circle
        ax.plot(0, 0, 'ro', markersize=15, label='Rest (i=0)')
    else:
        # Direction vectors
        color = 'blue' if i <= 4 else 'green'
        label = 'Cardinal' if i <= 4 and i > 0 else 'Diagonal' if i > 4 else None
        ax.arrow(0, 0, lattice.c[i, 0]*0.8, lattice.c[i, 1]*0.8,
                head_width=0.15, head_length=0.1, fc=color, ec=color,
                label=label if (i == 1 or i == 5) else None)
        
        # Add labels
        ax.text(lattice.c[i, 0]*1.1, lattice.c[i, 1]*1.1, str(i),
               fontsize=12, ha='center', va='center',
               bbox=dict(boxstyle='circle', facecolor='white', alpha=0.8))

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlabel('x-direction')
ax.set_ylabel('y-direction')
ax.set_title('D2Q9 Lattice Velocity Directions')
ax.legend()

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete")

## Visualization: Vorticity Field

In [ ]:
# Visualize vorticity field
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Velocity field
skip = 1
ax1.quiver(X[::skip, ::skip], Y[::skip, ::skip], 
          u[::skip, ::skip], v[::skip, ::skip])
ax1.set_aspect('equal')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Velocity Field (u = -y, v = x)')
ax1.grid(True, alpha=0.3)

# Vorticity field
im = ax2.contourf(X, Y, omega, levels=20, cmap='RdBu_r')
ax2.set_aspect('equal')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Vorticity Field (ω = ∂v/∂x - ∂u/∂y)')
plt.colorbar(im, ax=ax2, label='Vorticity')

plt.tight_layout()
plt.show()

print("\n✓ Vorticity visualization complete")

## VERIFICATION COMPLETE

All tests passed! The D2Q9 lattice implementation is verified and ready for use.